# Correct pickle timestamps

## Imports

In [1]:
from pathlib import Path
import re
import logging

import numpy as np
import pandas as pd
import pingouin
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm, Normalize
from matplotlib.cm import ScalarMappable

from gulp2p.preproc.tiff import Tiff
from gulp2p.preproc.experiment import Experiment, create_expt_df
from gulp2p.preproc import utils, imaging, behavior, trial as tr
from gulp2p.viz import plotting

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42
rcParams['svg.fonttype'] = "none"

### Logging config

In [2]:
log_level = logging.INFO    # log levels: DEBUG, INFO, WARNING, ERROR, CRITICAL
log_format = "{asctime} - {levelname} - {name} - [{filename} {funcName}() {lineno}] - {message}"
logging.basicConfig(format=log_format, level=log_level, style="{") 

logging.getLogger("gulp2p").setLevel(logging.DEBUG)
logger = logging.getLogger(__name__)

## Select Data

### Functions:

In [3]:
CELL_TYPES = ["DR015", "DR018", "DR019"]

GENETIC_TOOLS = ["iGluSnFR"]

### Create trial dataframe:

In [4]:
TIFF_FOLDER = Path(r"Z:\2PImaging\Kerstin\MIMS")

trial_df = tr.create_trial_df(TIFF_FOLDER, CELL_TYPES, GENETIC_TOOLS)

In [5]:
trial_df.head()

,path,date,line,cell_type,genetic_tool,fly,trial
0,Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_...,20231109,iGluSnFRxSS2232,iGluSnFR,SS2232,1,1
1,Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_...,20231109,iGluSnFRxSS2232,iGluSnFR,SS2232,1,2
2,Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_...,20231109,iGluSnFRxSS2232,iGluSnFR,SS2232,1,3
3,Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_...,20231109,iGluSnFRxSS2232,iGluSnFR,SS2232,1,4
4,Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_...,20231109,iGluSnFRxSS2232,iGluSnFR,SS2232,1,1


### Sort into experiments:

In [6]:
expt_df = create_expt_df(trial_df, CELL_TYPES, GENETIC_TOOLS)

2024-07-24 12:18:06,086 - INFO - gulp2p.preproc.utils - [utils.py load_trial() 639] - trial not processed: Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_iGluSnFRxSS2232_00001.tif
2024-07-24 12:18:06,094 - INFO - gulp2p.preproc.utils - [utils.py load_trial() 639] - trial not processed: Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_iGluSnFRxSS2232_00002.tif
2024-07-24 12:18:06,104 - INFO - gulp2p.preproc.utils - [utils.py load_trial() 639] - trial not processed: Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_iGluSnFRxSS2232_00003.tif
2024-07-24 12:18:06,106 - INFO - gulp2p.preproc.utils - [utils.py load_trial() 639] - trial not processed: Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_iGluSnFRxSS2232_00004.tif
2024-07-24 12:18:06,122 - INFO - gulp2p.preproc.utils - [utils.py load_trial() 639] - trial not processed: Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days_iGluSnFRxSS2232_Stripe_00001.tif
2024-07-24 12:18:06,129 - INFO - gulp2p.preproc.utils - [utils.py load_trial() 639] - trial not pr

In [7]:
expt_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   expt_name         40 non-null     object
 1   trial_paths       40 non-null     object
 2   date              40 non-null     object
 3   line              39 non-null     object
 4   cell_type         39 non-null     object
 5   genetic_tool      39 non-null     object
 6   fly               40 non-null     int64 
 7   trials            40 non-null     object
 8   proc_trial_count  40 non-null     int64 
dtypes: int64(2), object(7)
memory usage: 2.9+ KB


In [8]:
expt_df.tail()

,expt_name,trial_paths,date,line,cell_type,genetic_tool,fly,trials,proc_trial_count
35,20240626_DR015xiGluSnfR_Fly1,[Z:\2PImaging\Kerstin\MIMS\20240626\20240626_D...,20240626,DR015xiGluSnfR,DR015,iGluSnFR,1,[],0
36,20240626_DR015xiGluSnfR_Fly2,[Z:\2PImaging\Kerstin\MIMS\20240626\20240626_D...,20240626,DR015xiGluSnfR,DR015,iGluSnFR,2,[],0
37,20240702_DR015xiGluSnfR_Fly1,[Z:\2PImaging\Kerstin\MIMS\20240702\20240702_D...,20240702,DR015xiGluSnfR,DR015,iGluSnFR,1,[<gulp2p.preproc.trial.Trial object at 0x00000...,14
38,20240708_DR015xiGluSnfR_Fly1,[Z:\2PImaging\Kerstin\MIMS\20240708\DR015xiGlu...,20240708,DR015xiGluSnfR,DR015,iGluSnFR,1,[<gulp2p.preproc.trial.Trial object at 0x00000...,1
39,20240708_DR015xiGluSnfR_Fly2,[Z:\2PImaging\Kerstin\MIMS\20240708\DR015xiGlu...,20240708,DR015xiGluSnfR,DR015,iGluSnFR,2,[<gulp2p.preproc.trial.Trial object at 0x00000...,5


In [9]:
expt_df

,expt_name,trial_paths,date,line,cell_type,genetic_tool,fly,trials,proc_trial_count
0,20231109_iGluSnFRxSS2232_Fly1,[Z:\2PImaging\Kerstin\MIMS\20231109\Fly1_8days...,20231109,iGluSnFRxSS2232,iGluSnFR,SS2232,1,[],0
1,20231121_iGluSnFRxDR015_Fly1,[Z:\2PImaging\Kerstin\MIMS\20231121\iGluSnFRxD...,20231121,iGluSnFRxDR015,iGluSnFR,DR015,1,[],0
2,20231122_None_Fly1,[Z:\2PImaging\Kerstin\MIMS\20231122\file_00001...,20231122,None,None,None,1,[],0
3,20231128_DR019xiGluSnFr_Fly1,[Z:\2PImaging\Kerstin\MIMS\20231128\20231128_D...,20231128,DR019xiGluSnFr,DR019,iGluSnFR,1,[],0
4,20231128_DR019xiGluSnFr_Fly3,[Z:\2PImaging\Kerstin\MIMS\20231128\20231128_D...,20231128,DR019xiGluSnFr,DR019,iGluSnFR,3,[],0
5,20231129_DR018xiGluSnFr_Fly1,[Z:\2PImaging\Kerstin\MIMS\20231129\20231129_D...,20231129,DR018xiGluSnFr,DR018,iGluSnFR,1,[],0
6,20231129_DR018xiGluSnFr_Fly2,[Z:\2PImaging\Kerstin\MIMS\20231129\20231129_D...,20231129,DR018xiGluSnFr,DR018,iGluSnFR,2,[],0
7,20231204_DR019xEF024_Fly1,[Z:\2PImaging\Kerstin\MIMS\20231204\20231204_D...,20231204,DR019xEF024,DR019,EF024,1,[],0
8,20231204_DR019xEF024_Fly2,[Z:\2PImaging\Kerstin\MIMS\20231204\20231204_D...,20231204,DR019xEF024,DR019,EF024,2,[],0
9,20231204_DR019xEF024_Fly3,[Z:\2PImaging\Kerstin\MIMS\20231204\20231204_D...,20231204,DR019xEF024,DR019,EF024,3,[],0


## Correct timestamps

In [16]:
# For each trial:
#   get pickle name without timestamp prepended
#   Search for pickle files that contain the name
#       There should only be one
#   Rename the file with the new timestamp

for index, expt_df_row in expt_df.iterrows():
    
    for trial_path in expt_df_row['trial_paths']:
        tiff = Tiff(trial_path)
        pickle_path = utils.get_trial_pickle_path(trial_path, tiff.metadata['date'])

        pickle_date = pickle_path.name.split('-')[0]
        pickle_name_timestampless = '_'.join(pickle_path.name.split('_')[1:])
        search_key = f"{pickle_date}*{pickle_name_timestampless}"
        matching_pickles = list(pickle_path.parent.glob(search_key))

        if len(matching_pickles) == 0:
            # print(f"no pickle matches: {pickle_path}")
            continue

        if len(matching_pickles) != 1:
            # print(f"too many pickle matches: {pickle_path}")
            print(matching_pickles)
            continue

        if pickle_path == matching_pickles[0]:
            # print(f"pickle name matches: {pickle_path}")
            continue

        # Rename pickle file
        print(f"renaming {matching_pickles[0]} to {pickle_path}")
        matching_pickles[0].rename(pickle_path)


renaming Z:\GULP\Data\pickle\preproc\2024_07\20240708-124011_DR015xiGluSnfR_fly1_00002.pickle to Z:\GULP\Data\pickle\preproc\2024_07\20240708-104642_DR015xiGluSnfR_fly1_00002.pickle
renaming Z:\GULP\Data\pickle\preproc\2024_07\20240708-124023_DR015xiGluSnfR_fly1_00003.pickle to Z:\GULP\Data\pickle\preproc\2024_07\20240708-105612_DR015xiGluSnfR_fly1_00003.pickle
